# 62. Kivy Touch Apps — `kivy`
**Python Libraries Lab · Phase 3 · GUI, Games & Multimedia**

_Apps for touch screens and phones, written in Python._

💻 **Read and test in Colab, run the app on your laptop** (it opens a real window).

Kivy is a Python framework for apps with touch input, which can run on Windows, Mac, Linux, Android and iOS from the same code. It draws its own widgets with the graphics card, so apps look the same everywhere, and it understands taps, swipes and multi touch gestures.

**In this notebook:** You will build the ShopEasy quick order app. Big buttons to choose a product, plus and minus buttons for quantity, and a total that updates automatically. In Colab you test the logic and check the files. On your laptop you run the real app, and you can resize the window to see how it adapts like a phone screen.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
This library opens real windows, so it needs a computer with a screen. **In Colab** you read the code, check it, and (where possible) preview it without a window. **On your laptop** you run the real app: download the script from the lesson page or from Colab's Files panel, then follow the steps in the last cell. Kivy needs `pip install kivy` on your laptop. To put the app on an Android phone later, Kivy's Buildozer tool packages it (Linux or Mac).

## Step 1 · Logic you can test anywhere
Same rule as Tkinter: keep calculations separate from the screen.

In [ ]:
%%writefile quick_order.py
MENU = {"Steel Bottle": 499, "Yoga Mat": 1299, "Notebook": 149, "Desk Lamp": 899}

class Basket:
    def __init__(self):
        self.qty = {name: 0 for name in MENU}
    def add(self, name, n=1):
        self.qty[name] = max(0, self.qty[name] + n)
    def total(self):
        return sum(MENU[n] * q for n, q in self.qty.items())
    def summary(self):
        return ", ".join(f"{q} x {n}" for n, q in self.qty.items() if q) or "Basket is empty"

## Step 2 · Test it in Colab
Pure Python, so it runs anywhere.

In [ ]:
import importlib, quick_order
importlib.reload(quick_order)
from quick_order import Basket
b = Basket()
b.add("Yoga Mat"); b.add("Notebook", 3); b.add("Notebook", -1)
print(b.summary(), "| total Rs.", b.total())

## Step 3 · The KV layout
Describes the screen: a title, a grid of product rows and a total. `app.` refers to the Python app.

In [ ]:
%%writefile quickorder.kv
<ProductRow@BoxLayout>:
    name: ""
    size_hint_y: None
    height: "56dp"
    spacing: "8dp"
    Label:
        text: root.name
        font_size: "18sp"
    Button:
        text: "-"
        size_hint_x: None
        width: "56dp"
        on_release: app.change(root.name, -1)
    Label:
        text: str(app.counts.get(root.name, 0))
        size_hint_x: None
        width: "40dp"
        font_size: "20sp"
    Button:
        text: "+"
        size_hint_x: None
        width: "56dp"
        on_release: app.change(root.name, 1)

BoxLayout:
    orientation: "vertical"
    padding: "16dp"
    spacing: "10dp"
    Label:
        text: "ShopEasy quick order"
        font_size: "24sp"
        size_hint_y: None
        height: "48dp"
    BoxLayout:
        id: rows
        orientation: "vertical"
        spacing: "6dp"
    Label:
        text: app.summary
        color: 0.6, 0.65, 0.7, 1
    Label:
        text: "Total: Rs. " + str(app.total)
        font_size: "26sp"
        bold: True
        size_hint_y: None
        height: "56dp"

## Step 4 · The app class
Kivy properties (`counts`, `total`, `summary`) update the screen automatically when they change.

In [ ]:
%%writefile kivy_shop.py
from kivy.app import App
from kivy.factory import Factory
from kivy.lang import Builder
from kivy.properties import DictProperty, NumericProperty, StringProperty
from quick_order import MENU, Basket

class QuickOrderApp(App):
    counts = DictProperty()
    total = NumericProperty(0)
    summary = StringProperty("Basket is empty")

    def build(self):
        self.title = "ShopEasy quick order"
        self.basket = Basket()
        self.counts = dict(self.basket.qty)
        root = Builder.load_file("quickorder.kv")
        for name in MENU:
            root.ids.rows.add_widget(Factory.ProductRow(name=name))
        return root

    def change(self, name, step):
        self.basket.add(name, step)
        self.counts = dict(self.basket.qty)
        self.total = self.basket.total()
        self.summary = self.basket.summary()

if __name__ == "__main__":
    QuickOrderApp().run()

## Step 5 · Check the files, then run on your laptop
Validates both files. The KV file is parsed by Kivy's own parser when Kivy is installed.

In [ ]:
import py_compile
for f in ["quick_order.py", "kivy_shop.py"]:
    py_compile.compile(f, doraise=True)
text = open("quickorder.kv").read()
print("KV rules:", [l.strip() for l in text.splitlines() if l and not l.startswith(" ")])
print("All files valid. Laptop: pip install kivy, put quick_order.py, kivy_shop.py and quickorder.kv in one folder, then python kivy_shop.py")

## ✍️ Your turn
Make a new `Basket`, add 2 Steel Bottles and 1 Desk Lamp, and store the **total** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 1897, f"Expected 1897, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
b = Basket()
b.add("Steel Bottle", 2); b.add("Desk Lamp")
answer = b.total()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **App class**
- **Widget tree**
- **KV language**
- **Properties / binding**
- **Touch events**
- **Buildozer**

🎉 That's the last lesson: you've covered all 62 libraries!

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install kivy
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.
4. **Run the app:** download `quick_order.py`, `quickorder.kv`, `kivy_shop.py` (Colab: Files panel on the left → right-click → Download) into the `python-lab` folder, then run `python kivy_shop.py` in the same terminal.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.